In [5]:
import os
import numpy as np
import mitsuba as mi
import matplotlib.pyplot as plt
from sionna.rt import load_scene, Transmitter, PlanarArray, RadioMapSolver

# ================= 配置区域 =================
# height_list = [
#     1970.5, 1971.5, 1972.5, 1973.5, 1974.5, 1975.5, 1976.5, 1977.5, 
#     1979.5, 1980.5, 1981.5, 1982.5, 1983.5, 1984.5, 1985.5, 1986.5, 
#     1987.5, 1988.5, 1989.5, 1990.5, 1991.5, 1993.5, 1995.5, 1997.5, 2029.0
# ]

# save contents
save_dir = "sim_results"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# ================= 1. setting of scene and Transmitter =================
scene = load_scene('YNU/YNU_scene.xml')
scene.frequency = 3.45e9 
scene.tx_array = PlanarArray(num_rows=8,
                             num_cols=4,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern="tr38901",
                             polarization="cross")

# --- add the TX ---
tx_name = "21"
tx = Transmitter(name=tx_name,
                 position=[-181.02, -108.77, 2012],
                 orientation=[0.141, 0.2094, 0.0],
                 power_dbm=53.0)

# tx_name = "22"
# tx = Transmitter(name=tx_name,
#                  position=[-181.71, -109.31, 2012],
#                  orientation=[4.3298, 0.2094, 0.0],
#                  power_dbm=53.0)

# tx_name = "23"
# tx = Transmitter(name=tx_name,
#                  position=[-181.83, -108.45, 2012],
#                  orientation=[2.2354, 0.2094, 0.0],
#                  power_dbm=53.0)

scene.add(tx)
# ================= 2. 批量计算与保存循环 =================

rm_solver = RadioMapSolver()

# parameter of REM
map_center_x = -90
map_center_y = -113
map_size_x = 400
map_size_y = 400
map_cell_size = 1

# map_center_x = -123.7
# map_center_y = -123.6
# map_size_x = 512
# map_size_y = 512
# map_cell_size = 1

# original point
x = map_center_x - map_size_x / 2
y = map_center_y - map_size_y / 2

for i, h in enumerate(height_list):
    print(f"\n[{i+1}/{len(height_list)}] height Z = {h} m ...")
    
    # --- A. calculate REM ---
    rm = rm_solver(scene,
                   max_depth=5,            
                   samples_per_tx=10**9,   
                   cell_size=(map_cell_size, map_cell_size),
                   center=[map_center_x, map_center_y, h],
                   size=[map_size_x, map_size_y],    
                   orientation=[0, 0, 0])

    # --- B. Watt -> dBm ---
    power_watt = rm.rss[0].numpy()
    power_dbm = 10 * np.log10(np.maximum(power_watt, 1e-20) * 1000)
    power =power_dbm - 35.13
    power[power < -120] = -120 # threshold value

    # --- C. save .npz ---
    filename_npz = f"sim_data_h_{h}_{tx_name}.npz"
    save_path_npz = os.path.join(save_dir, filename_npz)
    np.savez(save_path_npz, 
             RSRP=power,
             X=x,
             Y=y,
             cell_size=map_cell_size,
             ALTITUDE=h)
    print(f"  -> NPZ has been saved: {save_path_npz}")

print("\n" + "="*40)
print("Simulation completed！")

场景已加载
场景频率已设置为: [3.45] GHz
TX 已添加到场景。
准备开始计算 1 个不同高度的覆盖图...

[1/1] 正在处理高度 Z = 2029.0 m ...
  -> NPZ 数据已保存: /home/winlab/桌面/sionna/height/sim_data_h_2029.0_21.npz

所有高度仿真完成！
